### Machine Learning model (Multi Classification)

In [12]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

#Model Selection
from sklearn.ensemble import RandomForestClassifier

#Mmodel Training
from sklearn.model_selection import KFold, cross_val_score, StratifiedKFold
from sklearn.model_selection import train_test_split

#Model Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV

#Model Evaluation
from sklearn.metrics import confusion_matrix,classification_report
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
from sklearn.metrics import RocCurveDisplay

In [13]:
df=pd.read_csv("data\shah_alam_labeled2.csv")
df.head(10)

,section,lat,lng,population,food_beverage,retail_outlet,service_business,entertainment,educational_inst,corporate_office,financial_inst,shopping_mall,automotive,healthcare,transportation,amenity_diversity_index,business_type
0,Section 1,3.0671,101.5017,1664,42,13,7,4,39,2,7,0,9,4,20,1.9114,Leisure & Lifestyle
1,Section 2,3.0718,101.5084,1991,53,32,33,6,45,25,7,3,15,6,20,2.1240,Leisure & Lifestyle
2,Section 3,3.0742,101.5097,1332,60,43,48,7,37,33,7,9,17,9,21,2.1644,Retail & Commerce
3,Section 4,3.0059,101.5511,851,32,10,11,2,5,3,0,0,23,2,6,1.8062,Community Services
4,Section 5,3.0831,101.5163,851,57,41,38,13,25,26,6,5,13,18,11,2.1737,Leisure & Lifestyle
5,Section 6,3.0821,101.5089,3033,56,26,25,9,22,11,2,0,14,8,12,2.0336,Leisure & Lifestyle
6,Section 7,3.0732,101.4924,44646,54,39,60,10,43,26,10,8,46,36,20,2.2326,Retail & Commerce
7,Section 8,3.0900,101.5088,7440,54,15,17,5,22,5,1,0,4,9,10,1.8809,Leisure & Lifestyle
8,Section 9,3.0832,101.5283,6403,55,40,60,11,23,35,10,6,23,19,11,2.1827,Retail & Commerce
9,Section 10,3.0803,101.5240,1072,60,49,60,14,26,50,21,7,23,48,19,2.2433,Business & Trade


### Data Exploration (EDA)

In [14]:
df["business_type"].value_counts()

business_type
Community Services     20
Leisure & Lifestyle    13
Retail & Commerce       9
Business & Trade        9
Food & Beverage         5
Name: count, dtype: int64

In [15]:
df["business_type"].value_counts().plot(kind="bar")

<Axes: xlabel='business_type'>

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   section                  56 non-null     object 
 1   lat                      56 non-null     float64
 2   lng                      56 non-null     float64
 3   population               56 non-null     int64  
 4   food_beverage            56 non-null     int64  
 5   retail_outlet            56 non-null     int64  
 6   service_business         56 non-null     int64  
 7   entertainment            56 non-null     int64  
 8   educational_inst         56 non-null     int64  
 9   corporate_office         56 non-null     int64  
 10  financial_inst           56 non-null     int64  
 11  shopping_mall            56 non-null     int64  
 12  automotive               56 non-null     int64  
 13  healthcare               56 non-null     int64  
 14  transportation           56 

In [17]:
df.describe()

,lat,lng,population,food_beverage,retail_outlet,service_business,entertainment,educational_inst,corporate_office,financial_inst,shopping_mall,automotive,healthcare,transportation,amenity_diversity_index
count,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000,56.000000
mean,3.083263,101.524812,14615.339286,42.375000,22.821429,23.250000,4.089286,18.053571,12.928571,4.339286,1.928571,24.428571,10.785714,12.071429,1.955996
std,0.058234,0.026184,15684.077727,14.359745,13.708799,17.998232,4.265194,9.502956,13.443291,5.660957,2.702572,11.703701,11.842473,6.954023,0.229005
min,2.979000,101.444300,851.000000,8.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.180300
25%,3.046450,101.508700,3887.000000,31.000000,12.000000,8.000000,1.000000,10.750000,3.750000,0.000000,0.000000,15.000000,2.000000,6.000000,1.824075
50%,3.077350,101.526300,11791.500000,43.000000,22.000000,18.500000,3.000000,19.500000,8.000000,3.000000,1.000000,24.000000,6.500000,13.000000,2.029900
75%,3.113050,101.543100,18589.000000,54.000000,32.500000,38.000000,6.000000,23.000000,19.250000,6.000000,3.000000,34.000000,14.250000,19.000000,2.115350
max,3.220300,101.571300,86548.000000,60.000000,49.000000,60.000000,22.000000,45.000000,59.000000,26.000000,13.000000,46.000000,48.000000,21.000000,2.267400


In [18]:
df.dtypes

section                     object
lat                        float64
lng                        float64
population                   int64
food_beverage                int64
retail_outlet                int64
service_business             int64
entertainment                int64
educational_inst             int64
corporate_office             int64
financial_inst               int64
shopping_mall                int64
automotive                   int64
healthcare                   int64
transportation               int64
amenity_diversity_index    float64
business_type               object
dtype: object

In [19]:
pd.crosstab(df.business_type,df.section)

section,Section 1,Section 10,Section 11,Section 12,Section 13,Section 14,Section 15,Section 16,Section 17,Section 18,...,Section U19,Section U2,Section U20,Section U3,Section U4,Section U5,Section U6,Section U7,Section U8,Section U9
business_type,,,,,,,,,,,,,,,,,,,,,
Business & Trade,0,1,1,1,0,1,1,0,0,1,...,1,1,0,0,0,0,0,0,0,0
Community Services,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,1,0,1
Food & Beverage,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Leisure & Lifestyle,1,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,1,0,0,0
Retail & Commerce,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,1,0,0,0,1,0


### Model Training

In [20]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_predict

x=df.drop("business_type",axis=1)
y=df.business_type

le=LabelEncoder()
y=le.fit_transform(y)

categorical_feature=x.select_dtypes(include=["object","category"]).columns

categorical_transformer=Pipeline(steps=[
    ("onehot",OneHotEncoder(handle_unknown="ignore"))
])

preprocessor=ColumnTransformer(
    transformers=[
        ("cat",categorical_transformer,categorical_feature)
    ], remainder="passthrough"
)

model_random=Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("model",RandomForestClassifier(random_state=42))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model_random, x, y, cv=skf, scoring='accuracy')

print(f"All Folds Accuracy: {scores}")
print(f"Mean Accuracy: {scores.mean() * 100:.2f}%")

y_pred_cv = cross_val_predict(model_random, x, y, cv=skf)  # ✅ gets predictions for every row

print("\nClassification Report (Cross-Validated):")
print(classification_report(y, y_pred_cv, target_names=le.classes_, zero_division=0))


All Folds Accuracy: [0.5        0.72727273 0.72727273 0.63636364 0.72727273]
Mean Accuracy: 66.36%

Classification Report (Cross-Validated):
                     precision    recall  f1-score   support

   Business & Trade       0.60      0.67      0.63         9
 Community Services       0.77      1.00      0.87        20
    Food & Beverage       1.00      0.20      0.33         5
Leisure & Lifestyle       0.67      0.62      0.64        13
  Retail & Commerce       0.29      0.22      0.25         9

           accuracy                           0.66        56
          macro avg       0.66      0.54      0.54        56
       weighted avg       0.66      0.66      0.63        56



### Model Tuning

In [21]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint


param_dist = {
    "model__n_estimators":      [50, 100, 200, 300],
    "model__max_depth":         [None, 5, 10, 15, 20],
    "model__min_samples_split": [2, 4, 6, 8],
    "model__min_samples_leaf":  [1, 2, 3],
    "model__max_features":      ["sqrt", "log2", None],
    "model__bootstrap":         [True, False],
}


random_search = RandomizedSearchCV(
    model_random, 
    param_distributions=param_dist, 
    n_iter=10, 
    cv=skf, 
    scoring='accuracy', 
    n_jobs=-1, 
    random_state=42
)


random_search.fit(x, y)


print(f"Best Accuracy: {random_search.best_score_ * 100:.2f}%")
print(f"Best Params: {random_search.best_params_}")
y_pred_best = cross_val_predict(random_search.best_estimator_, x, y, cv=skf)
print(classification_report(y, y_pred_best, target_names=le.classes_, zero_division=0))

Best Accuracy: 66.21%
Best Params: {'model__n_estimators': 200, 'model__min_samples_split': 6, 'model__min_samples_leaf': 2, 'model__max_features': None, 'model__max_depth': 10, 'model__bootstrap': False}
                     precision    recall  f1-score   support

   Business & Trade       0.75      0.67      0.71         9
 Community Services       0.81      0.85      0.83        20
    Food & Beverage       0.50      0.40      0.44         5
Leisure & Lifestyle       0.45      0.38      0.42        13
  Retail & Commerce       0.58      0.78      0.67         9

           accuracy                           0.66        56
          macro avg       0.62      0.62      0.61        56
       weighted avg       0.65      0.66      0.65        56

